# VIBEJOBBER

Serper job search → **`JOB_POSTINGS`** list → agents: Job Hunter → page fetch → cover letter → CV → form-fill **plan** (JSON only; no ATS POST). Artifacts: `outputs/` next to this notebook (`Path.cwd()`).

In [6]:
import asyncio
import hashlib
import json
import os
import re
from pathlib import Path
from typing import Annotated

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

load_dotenv(override=True)
print("[init] dotenv loaded")

[init] dotenv loaded


In [7]:
# --- paths & global job list -------------------------------------------------

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[init] OUTPUT_DIR={OUTPUT_DIR}")

JOB_POSTINGS: list[dict] = []


def _short_job_id(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:12]


JOB_BOARD_SITES = (
    "site:lever.co | site:greenhouse.io | site:jobs.ashbyhq.com | "
    "site:app.dover.io | site:workable.com"
)


def build_job_search_query(job_title: str, or_terms: list[str] | None = None) -> str:
    title = job_title.strip()
    if not title:
        raise ValueError("job_title must be non-empty")
    variants: list[str] = [title]
    lower = title.lower()
    if "developer" not in lower:
        variants.append(f"{title} developer")
    if "engineer" not in lower:
        variants.append(f"{title} engineer")
    if or_terms:
        for extra in or_terms:
            e = extra.strip()
            if e and e not in variants:
                variants.append(e)
    or_clause = " | ".join(variants)
    return f"{JOB_BOARD_SITES} ({or_clause})"


def search_jobs(
    job_title: str,
    *,
    num: int = 10,
    or_terms: list[str] | None = None,
    hl: str = "en",
    gl: str | None = None,
    past_week: bool = True,
) -> dict:
    print(f"[search_jobs] title={job_title!r} num={num} past_week={past_week}")
    api_key = os.getenv("SERPER_API_KEY")
    if not api_key:
        raise RuntimeError("SERPER_API_KEY is not set")

    q = build_job_search_query(job_title, or_terms=or_terms)
    payload: dict = {"q": q, "num": num, "hl": hl, "autocorrect": True}
    if gl:
        payload["gl"] = gl
    if past_week:
        payload["tbs"] = "qdr:w"

    print(f"[search_jobs] POST google.serper.dev/search q={q[:120]}...")
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    n = len(data.get("organic") or [])
    print(f"[search_jobs] done, organic results={n}")
    return data


def ingest_serper_results(data: dict, *, replace: bool = True) -> int:
    organic = data.get("organic") or []
    if replace:
        print(f"[ingest] clearing JOB_POSTINGS (had {len(JOB_POSTINGS)} rows)")
        JOB_POSTINGS.clear()
    added = 0
    for o in organic:
        link = (o.get("link") or "").strip()
        if not link:
            continue
        jid = _short_job_id(link)
        row = {
            "id": jid,
            "title": o.get("title") or "",
            "link": link,
            "snippet": o.get("snippet") or "",
            "page_text": None,
        }
        JOB_POSTINGS.append(row)
        added += 1
        print(f"[ingest] + job id={jid} title={row['title'][:70]!r}")
    print(f"[ingest] added={added} total_in_list={len(JOB_POSTINGS)}")
    return added

[init] OUTPUT_DIR=/Users/Apple/vscode_projects/vibejobber/outputs


In [8]:
# --- Tools -------------------------------------------------------------------


@function_tool
def search_job_boards(
    job_title: Annotated[str, "Core role or stack, e.g. Flutter, Staff Backend"],
    extra_keywords: Annotated[
        str,
        "Optional comma-separated OR terms, e.g. 'mobile developer, iOS'",
    ] = "",
    num_results: Annotated[int, "How many Google results to return"] = 10,
    past_week: Annotated[bool, "Restrict to past week in Google"] = True,
) -> str:
    print("[tool:search_job_boards] invoked")
    extras = [s.strip() for s in extra_keywords.split(",") if s.strip()] or None
    data = search_jobs(job_title, num=num_results, or_terms=extras, past_week=past_week)
    ingest_serper_results(data, replace=True)
    slim = [
        {"id": j["id"], "title": j["title"], "link": j["link"], "snippet": j["snippet"]}
        for j in JOB_POSTINGS
    ]
    out = {
        "search_query": data.get("searchParameters", {}).get("q"),
        "stored_jobs": slim,
        "count": len(slim),
    }
    print(f"[tool:search_job_boards] returning count={len(slim)}")
    return json.dumps(out, indent=2)


def _html_to_text(html: str, max_chars: int = 100_000) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n[truncated]"
    return text


@function_tool
def fetch_job_page(
    job_index: Annotated[int, "0-based index into JOB_POSTINGS (same order as search)"],
) -> str:
    print(f"[tool:fetch_job_page] invoked job_index={job_index} list_len={len(JOB_POSTINGS)}")
    if job_index < 0 or job_index >= len(JOB_POSTINGS):
        err = {"error": "job_index out of range", "len": len(JOB_POSTINGS)}
        print(f"[tool:fetch_job_page] ERROR {err}")
        return json.dumps(err)

    job = JOB_POSTINGS[job_index]
    url = job["link"]
    print(f"[tool:fetch_job_page] GET {url}")
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml",
    }
    try:
        r = requests.get(url, headers=headers, timeout=35)
        print(f"[tool:fetch_job_page] status={r.status_code} bytes={len(r.content)}")
        r.raise_for_status()
        text = _html_to_text(r.text)
        job["page_text"] = text
        print(f"[tool:fetch_job_page] extracted page_text chars={len(text)}")
    except Exception as e:
        job["page_text"] = f"[fetch failed: {e}]"
        print(f"[tool:fetch_job_page] FAILED: {e}")
        return json.dumps({"ok": False, "job_id": job["id"], "error": str(e)})

    preview = text[:4000]
    return json.dumps(
        {"ok": True, "job_id": job["id"], "title": job["title"], "text_preview": preview},
        indent=2,
    )


def _write_txt(path: Path, body: str) -> None:
    path.write_text(body, encoding="utf-8")
    print(f"[io] wrote text file {path} ({len(body)} chars)")


def _write_pdf(path: Path, body: str) -> None:
    try:
        from fpdf import FPDF  # pip install fpdf2
    except ImportError as e:
        raise ImportError("Install PDF support: pip install fpdf2") from e

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)
    safe = body.encode("latin-1", errors="replace").decode("latin-1")
    for line in safe.splitlines():
        pdf.multi_cell(0, 6, line)
    pdf.output(str(path))
    print(f"[io] wrote PDF {path}")


@function_tool
def save_cover_letter(
    job_index: Annotated[int, "Index in JOB_POSTINGS"],
    letter_body: Annotated[str, "Full cover letter text"],
    file_format: Annotated[str, "txt or pdf — match what the posting / form asks for"],
) -> str:
    print(f"[tool:save_cover_letter] job_index={job_index} format={file_format!r}")
    if job_index < 0 or job_index >= len(JOB_POSTINGS):
        return json.dumps({"error": "bad job_index"})
    job = JOB_POSTINGS[job_index]
    fmt = file_format.lower().strip().lstrip(".")
    if fmt not in ("txt", "pdf"):
        print(f"[tool:save_cover_letter] invalid format {fmt!r}, defaulting to txt")
        fmt = "txt"
    path = OUTPUT_DIR / f"{job['id']}_cover_letter.{fmt}"
    if fmt == "txt":
        _write_txt(path, letter_body)
    else:
        _write_pdf(path, letter_body)
    job["cover_letter_path"] = str(path)
    job["cover_letter_format"] = fmt
    print(f"[tool:save_cover_letter] done path={path}")
    return json.dumps({"saved": str(path), "format": fmt, "job_id": job["id"]})


@function_tool
def save_cv_for_job(
    job_index: Annotated[int, "Index in JOB_POSTINGS"],
    cv_body: Annotated[str, "CV content tailored to the role (plain or markdown)"],
    file_format: Annotated[str, "txt or pdf"],
) -> str:
    print(f"[tool:save_cv_for_job] job_index={job_index} format={file_format!r}")
    if job_index < 0 or job_index >= len(JOB_POSTINGS):
        return json.dumps({"error": "bad job_index"})
    job = JOB_POSTINGS[job_index]
    fmt = file_format.lower().strip().lstrip(".")
    if fmt not in ("txt", "pdf"):
        fmt = "txt"
    path = OUTPUT_DIR / f"{job['id']}_cv.{fmt}"
    if fmt == "txt":
        _write_txt(path, cv_body)
    else:
        _write_pdf(path, cv_body)
    job["cv_path"] = str(path)
    job["cv_format"] = fmt
    print(f"[tool:save_cv_for_job] done path={path}")
    return json.dumps({"saved": str(path), "format": fmt, "job_id": job["id"]})


@function_tool
def save_form_fill_plan(
    job_index: Annotated[int, "Index in JOB_POSTINGS"],
    plan_json: Annotated[
        str,
        'JSON string: {"fields":[{"name":"...","value":"..."}], "notes":"..."}',
    ],
) -> str:
    print(f"[tool:save_form_fill_plan] job_index={job_index}")
    if job_index < 0 or job_index >= len(JOB_POSTINGS):
        return json.dumps({"error": "bad job_index"})
    job = JOB_POSTINGS[job_index]
    try:
        plan = json.loads(plan_json)
    except json.JSONDecodeError as e:
        print(f"[tool:save_form_fill_plan] JSON error: {e}")
        return json.dumps({"error": f"invalid JSON: {e}"})

    fields = plan.get("fields") or []
    print(f"[tool:save_form_fill_plan] {len(fields)} fields:")
    for i, f in enumerate(fields):
        name = f.get("name", "?")
        val = f.get("value", "")
        preview = (val[:120] + "…") if len(str(val)) > 120 else val
        print(f"  [{i}] {name!r} => {preview!r}")

    path = OUTPUT_DIR / f"{job['id']}_form_fill_plan.json"
    path.write_text(json.dumps(plan, indent=2), encoding="utf-8")
    job["form_plan_path"] = str(path)
    print(f"[tool:save_form_fill_plan] wrote {path}")
    return json.dumps({"saved": str(path), "job_id": job["id"], "field_count": len(fields)})

In [9]:
# --- Agents & runners --------------------------------------------------------

job_hunter_agent = Agent(
    name="Job Hunter",
    instructions=(
        "You find roles on ATS boards (Lever, Greenhouse, Ashby, Dover, Workable). "
        "Call search_job_boards with a focused job_title and optional extra_keywords. "
        "Jobs are stored in an internal list automatically. Summarize stored_jobs with markdown links."
    ),
    tools=[search_job_boards],
    model="gpt-4o-mini",
)

job_page_fetch_agent = Agent(
    name="Job Page Fetcher",
    instructions=(
        "You load one job posting into memory for downstream steps. "
        "The user message includes job_index (integer). Call fetch_job_page exactly once with that index. "
        "Then briefly confirm you fetched the page and mention if the preview looks like a real job description."
    ),
    tools=[fetch_job_page],
    model="gpt-4o-mini",
)

cover_letter_agent = Agent(
    name="Cover Letter Writer",
    instructions=(
        "You write a tailored cover letter. The user message includes: job_index, candidate_profile, "
        "and an excerpt of the job page. Decide file_format: use 'pdf' if the page mentions PDF upload "
        "or 'submit as PDF'; otherwise use 'txt'. Write the full letter, then call save_cover_letter "
        "with job_index, the full letter text, and file_format."
    ),
    tools=[save_cover_letter],
    model="gpt-4o-mini",
)

cv_agent = Agent(
    name="CV Writer",
    instructions=(
        "You produce a concise CV tailored to the job requirements. The user message includes job_index, "
        "candidate_profile, and job excerpt. Highlight matching skills and experience. "
        "Use the same file_format rule as cover letters (pdf vs txt from posting language). "
        "Call save_cv_for_job with job_index, full cv_body, and file_format."
    ),
    tools=[save_cv_for_job],
    model="gpt-4o-mini",
)

form_filler_agent = Agent(
    name="Form Fill Planner",
    instructions=(
        "You plan ATS application form answers (do not claim you submitted anything). "
        "From the job excerpt + candidate_profile, build a JSON object with keys 'fields' (list of "
        "{'name','value'}) and optional 'notes'. Typical fields: name, email, phone, LinkedIn, "
        "years of experience, work authorization, cover letter path. "
        "Call save_form_fill_plan with job_index and plan_json as a compact JSON string."
    ),
    tools=[save_form_fill_plan],
    model="gpt-4o-mini",
)


async def run_job_hunter(message: str) -> str:
    print(f"[run_job_hunter] message={message[:200]!r}...")
    result = await Runner.run(job_hunter_agent, message)
    print("[run_job_hunter] finished")
    return result.final_output


def _job_excerpt(job_index: int, max_chars: int = 14_000) -> str:
    job = JOB_POSTINGS[job_index]
    t = job.get("page_text") or job.get("snippet") or ""
    return t[:max_chars]


async def run_sequential_application_pipeline(
    job_index: int,
    candidate_profile: str,
) -> dict[str, str]:
    print("\n" + "=" * 60)
    print(f"[pipeline] START job_index={job_index} jobs_in_memory={len(JOB_POSTINGS)}")
    print("=" * 60 + "\n")

    if job_index < 0 or job_index >= len(JOB_POSTINGS):
        raise IndexError(f"job_index {job_index} invalid; list has {len(JOB_POSTINGS)} jobs")

    j = JOB_POSTINGS[job_index]
    print(f"[pipeline] job id={j['id']} link={j['link']}")

    print("\n>>> [pipeline] STEP 1: Job Page Fetcher agent\n")
    r_fetch = await Runner.run(
        job_page_fetch_agent,
        f"job_index={job_index}. Fetch this job posting.",
    )
    print(f"[pipeline] fetch agent final_output (excerpt):\n{r_fetch.final_output[:800]!r}\n")

    excerpt = _job_excerpt(job_index)

    print("\n>>> [pipeline] STEP 2: Cover Letter agent\n")
    r_cover = await Runner.run(
        cover_letter_agent,
        f"job_index={job_index}\n\ncandidate_profile:\n{candidate_profile}\n\n"
        f"job_page_excerpt:\n{excerpt}\n",
    )
    print(f"[pipeline] cover agent final_output (excerpt):\n{r_cover.final_output[:800]!r}\n")

    print("\n>>> [pipeline] STEP 3: CV agent\n")
    r_cv = await Runner.run(
        cv_agent,
        f"job_index={job_index}\n\ncandidate_profile:\n{candidate_profile}\n\n"
        f"job_page_excerpt:\n{excerpt}\n",
    )
    print(f"[pipeline] cv agent final_output (excerpt):\n{r_cv.final_output[:800]!r}\n")

    print("\n>>> [pipeline] STEP 4: Form fill planner agent\n")
    r_form = await Runner.run(
        form_filler_agent,
        f"job_index={job_index}\n\ncandidate_profile:\n{candidate_profile}\n\n"
        f"job_page_excerpt:\n{excerpt}\n",
    )
    print(f"[pipeline] form agent final_output (excerpt):\n{r_form.final_output[:800]!r}\n")

    print("=" * 60)
    print("[pipeline] DONE")
    print("=" * 60 + "\n")

    return {
        "fetch": r_fetch.final_output,
        "cover_letter": r_cover.final_output,
        "cv": r_cv.final_output,
        "form_plan": r_form.final_output,
    }


async def run_search_then_pipeline_first_job(
    search_message: str,
    candidate_profile: str,
) -> dict[str, str]:
    print("[run_search_then_pipeline_first_job] phase: search")
    await run_job_hunter(search_message)
    if not JOB_POSTINGS:
        print("[run_search_then_pipeline_first_job] no jobs stored; abort")
        return {}
    print("[run_search_then_pipeline_first_job] phase: pipeline on index 0")
    return await run_sequential_application_pipeline(0, candidate_profile)

print("[init] agents and runners defined")

[init] agents and runners defined


In [10]:
# --- Run examples (uncomment) ------------------------------------------------
# Run cells top → bottom once per kernel restart.

CANDIDATE = """
Name: Alex Dev
Email: alex@example.com
5 years Flutter & Dart, iOS/Android releases, REST, Firebase.
"""

summary = await run_job_hunter(
    "Find Flutter mobile developer jobs from the last week; extra_keywords: mobile developer"
)
print(summary)
print("Stored jobs:", len(JOB_POSTINGS), JOB_POSTINGS[0]["title"] if JOB_POSTINGS else None)

# out = await run_sequential_application_pipeline(0, CANDIDATE)
# print({k: v[:400] for k, v in out.items()})

out = await run_search_then_pipeline_first_job(
    "Find Flutter mobile developer jobs from the last week; extra_keywords: mobile developer",
    CANDIDATE,
)
print(out)

for i in range(len(JOB_POSTINGS)):
    await run_sequential_application_pipeline(i, CANDIDATE)

print("(Uncomment examples above after running prior cells.)")

[run_job_hunter] message='Find Flutter mobile developer jobs from the last week; extra_keywords: mobile developer'...
[tool:search_job_boards] invoked
[search_jobs] title='Flutter mobile developer' num=10 past_week=True
[search_jobs] POST google.serper.dev/search q=site:lever.co | site:greenhouse.io | site:jobs.ashbyhq.com | site:app.dover.io | site:workable.com (Flutter mobile devel...
[search_jobs] done, organic results=10
[ingest] clearing JOB_POSTINGS (had 0 rows)
[ingest] + job id=897612371c6c title='Resilient Co - Flutter Frontend Engineer Sr - Lever'
[ingest] + job id=7d48bea20df8 title='Senior/Staff Flutter Engineer (Identity Defender) - Greenhouse'
[ingest] + job id=2a62dfb399f5 title='Full Stack Mobile Application Developer - Two95 International Inc.'
[ingest] + job id=baa7be7f39d2 title='Software Developer for Mobile and Web Apps | Twin Sun'
[ingest] + job id=7f6e47dccb0d title='Senior Automation Tester - Flutter, Agentic AI & API Systems | Zippd'
[ingest] + job id=1f3ba44c8

[non-fatal] Tracing: request failed: [SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2648)


[tool:save_cv_for_job] job_index=0 format='txt'
[io] wrote text file /Users/Apple/vscode_projects/vibejobber/outputs/897612371c6c_cv.txt (2315 chars)
[tool:save_cv_for_job] done path=/Users/Apple/vscode_projects/vibejobber/outputs/897612371c6c_cv.txt
[pipeline] cv agent final_output (excerpt):
'Your CV has been successfully tailored and saved for the job as a Flutter Frontend Engineer Sr at Resilient Co. \n\nHere are the details:\n\n- **File Name**: 897612371c6c_cv.txt  \n- **File Format**: TXT  \n\nIf you need any further modifications or assistance, feel free to ask!'


>>> [pipeline] STEP 4: Form fill planner agent

[tool:save_form_fill_plan] job_index=0
[tool:save_form_fill_plan] 4 fields:
  [0] 'name' => 'Alex Dev'
  [1] 'email' => 'alex@example.com'
  [2] 'years of experience' => '5'
  [3] 'work authorization' => 'N/A'
[tool:save_form_fill_plan] wrote /Users/Apple/vscode_projects/vibejobber/outputs/897612371c6c_form_fill_plan.json
[pipeline] form agent final_output (excerpt):
"I'

CancelledError: 